In [18]:
import cv2
import pytesseract
import re
import ultralytics
import numpy as np
import pandas as pd

In [19]:
model_path=ultralytics.YOLO("best (1).pt")

In [20]:
pytesseract.pytesseract.tesseract_cmd=r"C:\Program Files\Tesseract-OCR\tesseract.exe"

In [21]:
vehicle_data = {
    "Registration_Number": [
        "MH12AB3456",
        "TN12AX5610",
        "KL07CD1234",
        "KA03EF5678",
        "TN09GH4321",
    ],
    "Owner_Name": [
        "Rahul Sharma",
        "Arun Kumar",
        "Anjali Nair",
        "Priya Rao",
        "Vijay Kumar"
    ],
    "House_Name": [
        "Green Villa",
        "Sunrise House",
        "Rose Villa",
        "Lake View",
        "Hill House"
    ],
    "Place": [
        "Pune",
        "Chennai",
        "Kochi",
        "Bengaluru",
        "Chennai"
    ],
    "Phone_Number": [
        "0000000001",
        "0000000002",
        "0000000003",
        "0000000004",
        "0000000005"
    ]
}


In [22]:
vehicle_df=pd.DataFrame(vehicle_data)

In [23]:
img=cv2.imread("num_plate5.jpeg")

In [24]:
results=model_path.predict(source=img)


0: 448x640 1 license_plate, 6424.4ms
Speed: 307.2ms preprocess, 6424.4ms inference, 86.6ms postprocess per image at shape (1, 3, 448, 640)


In [25]:
for result in results:

    for box in result.boxes:

        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)

        height, width = img.shape[:2]

        x1 = max(0, x1)
        y1 = max(0, y1)
        x2 = min(width, x2)
        y2 = min(height, y2)

        plate = img[y1:y2, x1:x2].copy()

        if plate.size == 0:
            continue

        cv2.imshow("Cropped Number Plate", plate)
        cv2.waitKey(1000)
        cv2.destroyAllWindows()
        gray = cv2.cvtColor(plate, cv2.COLOR_BGR2GRAY)

        gray = cv2.resize(
            gray, None, fx=3, fy=3,
            interpolation=cv2.INTER_CUBIC
        )

        text = pytesseract.image_to_string(
            gray,
            config="--psm 7 -c tessedit_char_whitelist=ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789"
        )

        registration_text = re.sub(
            r"[^A-Z0-9]", "", text.upper()
        )

        pattern = r"^[A-Z]{2}\d{1,2}[A-Z]{1,3}\d{4}$"

        if re.fullmatch(pattern, registration_text):
            registration_number = registration_text
            match = vehicle_df[
                vehicle_df["Registration_Number"]
                == registration_number
            ]

            # 11. Display the matching vehicle details
            if not match.empty:

                vehicle = match.iloc[0]

                print("\nVehicle Found")
                print("----------------------------")
                print(
                    "Registration Number:",
                    vehicle["Registration_Number"]
                )
                print(
                    "Owner Name:",
                    vehicle["Owner_Name"]
                )
                print(
                    "House Name:",
                    vehicle["House_Name"]
                )
                print(
                    "Place:",
                    vehicle["Place"]
                )
                print(
                    "Phone Number:",
                    vehicle["Phone_Number"]
                )

                display_text = registration_number + " - Found"

            else:
                print("\nVehicle Not Found in Database")

                display_text = (
                    registration_number + " - Not Found"
                )


        else:
            registration_number = "OCR Failed"

        print("Raw OCR:", repr(text))
        print("Cleaned OCR:", registration_text)
        print("Registration Number:", registration_number)
        cv2.rectangle(
            img, (x1, y1), (x2, y2),
            (0, 0, 255), 3
        )

        cv2.putText(
            img,
            registration_number,
            (x1, max(25, y1 - 10)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (0, 0, 255),
            2
        )

cv2.imshow("YOLO + OCR Result", img)
cv2.imwrite("result.jpg", img)
cv2.waitKey(0)
cv2.destroyAllWindows()


Vehicle Found
----------------------------
Registration Number: MH12AB3456
Owner Name: Rahul Sharma
House Name: Green Villa
Place: Pune
Phone Number: 0000000001
Raw OCR: 'MH12AB3456\n'
Cleaned OCR: MH12AB3456
Registration Number: MH12AB3456


In [26]:
# for result in results:

#     if result.boxes is None or len(result.boxes) == 0:
#         continue

#     for box in result.boxes:

#         # Get bounding box coordinates from YOLO
#         x1, y1, x2, y2 = (
#             box.xyxy[0].cpu().numpy().astype(int)
#         )

#         yolo_conf = float(box.conf[0].item())

#         # Ensure coordinates remain inside the image
#         height, width = img.shape[:2]
#         pad_x=int((x2-x1)*0.02)
#         pad_y=int((y2-y1)*0.05)
#         crop_x1 = max(0, x1 + pad_x)
#         crop_x2 = min(width, x2 - pad_x)
#         crop_y1 = max(0, y1 + pad_y)
#         crop_y2 = min(height, y2 - pad_y)

#         plate = img[crop_y1:crop_y2, crop_x1:crop_x2].copy()
#         # x1 = max(0, min(x1, width - 1))
#         # x2 = max(0, min(x2, width))
#         # y1 = max(0, min(y1, height - 1))
#         # y2 = max(0, min(y2, height))

#         # if x2 <= x1 or y2 <= y1:
#         #     continue
#         # plate = img[y1:y2, x1:x2].copy()
#         if plate.size == 0:
#             continue
#         cv2.imshow("Cropped plate",plate)
#         cv2.waitKey(40000)
#         cv2.destroyAllWindows()

#         print(f"YOLO confidence: {yolo_conf:.3f}")
#         gray=cv2.cvtColor(plate,cv2.COLOR_BGR2GRAY)
#         gray = cv2.resize(
#             gray,
#             None,
#             fx=3,
#             fy=3,
#             interpolation=cv2.INTER_CUBIC
#         )
#         gray = cv2.bilateralFilter(
#             gray,
#             9,
#             75,
#             75
#         )
#         enhanced = cv2.createCLAHE(
#             clipLimit=2.0,
#             tileGridSize=(8, 8)
#         ).apply(gray)
#         _, thresholded = cv2.threshold(
#             enhanced,
#             0,
#             255,
#             cv2.THRESH_BINARY + cv2.THRESH_OTSU
#         )
#         pattern = r"[A-Z]{2}\d{1,2}[A-Z]{1,3}\d{4}"
#         OCR_CONFIGS = [
#             "--oem 3 --psm 7 -c tessedit_char_whitelist=ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789",
#             "--oem 3 --psm 8 -c tessedit_char_whitelist=ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789"
#         ]
#         candidates = []
#         for processed_image in [enhanced, thresholded]:
#             for config in OCR_CONFIGS:

#                 text = pytesseract.image_to_string(
#                     processed_image,
#                     config=config
#                 )

#                 registration_text = re.sub(
#                     r"[^A-Z0-9]",
#                     "",
#                     text.upper()
#                 )

#                 match = re.search(pattern, registration_text)

#                 print("Raw OCR:", repr(text))
#                 print("Cleaned OCR:", registration_text)

#                 if match:
#                     candidates.append(match.group())

#         # Remove duplicate candidates while preserving order
#         candidates = list(dict.fromkeys(candidates))

#         if len(candidates) == 1:
#             registration_number = candidates[0]

#         elif len(candidates) > 1:
#             print("Different valid-format OCR candidates:", candidates)
#             registration_number = ""

#         else:
#             registration_number = ""

#         print(
#             "Extracted Registration Number:",
#             registration_number or "No reliable format match"
#         )
#         # text=pytesseract.image_to_string(enhanced)
#         # registration_text = re.sub(
#         #     r"[^A-Z0-9]",
#         #     "",
#         #     text.upper()
#         # )
#         # # pattern = r"[A-Z]{2}\d{1,2}[A-Z]{1,3}\d{4}"
#         # # match=re.search(pattern, registration_text)
#         # # registration_number = match.group()
        
#         # pattern = r"[A-Z]{2}\d{1,2}[A-Z]{1,3}\d{4}"

#         # match = re.search(pattern, registration_text)

#         # if match is not None:
#         #     registration_number = match.group()
#         # else:
#         #     registration_number = ""

#         # print("Raw OCR text:", repr(text))
#         # print("Cleaned OCR text:", repr(registration_text))

#         # print(
#         #     "Extracted Registration Number:",
#         #     registration_number or "No valid registration number recognized"
#         # )

#         # print(
#         #     "Extracted Registration Number:",
#         #     registration_number or "No text recognized"
#         # )
#         data=pytesseract.image_to_data(enhanced,output_type=pytesseract.Output.DICT)
#         for i in range(len(data["text"])):

#             word = data["text"][i].strip()

#             try:
#                 ocr_conf = float(data["conf"][i])
#             except (ValueError, TypeError):
#                 ocr_conf = -1

#             if word and ocr_conf >= 0:
#                 print(
#                     f"OCR word: {word}, "
#                     f"confidence: {ocr_conf:.1f}"
#                 )
#         cv2.rectangle(
#             img,
#             (x1, y1),
#             (x2, y2),
#             (0, 0, 255),
#             3
#         )

#         label = registration_number or "OCR Failed"

#         # Put text above the plate when possible
#         text_y = max(25, y1 - 10)

#         cv2.putText(
#             img,
#             label,
#             (x1, text_y),
#             cv2.FONT_HERSHEY_SIMPLEX,
#             0.8,
#             (0, 0, 255),
#             2
#         )

# cv2.imshow("Plate Detection",img)
# saved=cv2.imwrite("result.jpg",img)
# cv2.waitKey(40000)
# cv2.destroyAllWindows()

